In [1]:
import pandas as pd
import pickle
from sklearn.preprocessing import OneHotEncoder

with open('best_model_top_28_features.pkl', 'rb') as f:
    model = pickle.load(f)

In [2]:
df_demo = pd.read_csv("sample_data/demographic.csv")
df_hosp = pd.read_csv("sample_data/hospital.csv")
df_phys = pd.read_csv("sample_data/physiological.txt", sep="\t")
df_severity = pd.read_json("sample_data/severity.json")

In [3]:
df_severity = df_severity.explode(list(df_severity.columns[2:]))
df_severity.reset_index(drop=True, inplace=True)
demographic_hospital_df = df_demo.merge(df_hosp, on='pasient_id', how='left')
demographic_hospital_physiological_df = demographic_hospital_df.merge(df_phys, on='pasient_id', how='left')
df = demographic_hospital_physiological_df.merge(df_severity, on='pasient_id', how='left')

pasient_ids = df[['pasient_id']]

In [4]:
df.rename(columns={"sykdom_underkategori": "sykdomskategori"}, inplace=True)
df.drop(columns=["sykdomskategori_id", 'sykehusdød', 'dødsfall', 'dnr_dag', 'bilirubin', 'adl_pasient'], inplace=True)

In [5]:
fill_values = {
    'serumalbumin': 3.5,
    'lungefunksjon': 333.3,
    'kreatinin': 1.01,
    'blodurea_nitrogen': 6.51,
    'hvite_blodlegemer': 9,
    'urinmengde': 2502,
    'blod_ph': 7.4
}
df.fillna(fill_values, inplace=True)

In [6]:
inntekt_mapping = {
    'under $11k': 0, 
    '$11-$25k': 1,
    '$25-$50k': 2, 
    '>$50k': 3
}
df['inntekt'] = df['inntekt'].map(inntekt_mapping)

kreft_mapping = {
    'no': 0, 
    'yes': 1,
    'metastatic': 2
}
df['kreft'] = df['kreft'].map(kreft_mapping)

dnr_mapping = {
    'none': 0,
    'dnr ved innleggelse': 1, 
    'dnr før innleggelse': 2
}
df['dnr_status'] = df['dnr_status'].map(dnr_mapping)

In [7]:
kjønn_encoder = OneHotEncoder(sparse_output=False, drop='first')
etnisitet_encoder = OneHotEncoder(sparse_output=False)

kjønn_encoder.fit(df[['kjønn']])
etnisitet_encoder.fit(df[['etnisitet']])

def kjønn_dummies(df):
    kjønn_encoded = kjønn_encoder.transform(df[['kjønn']])
    encoded_df = pd.DataFrame(kjønn_encoded, columns=kjønn_encoder.get_feature_names_out(['kjønn']), index=df.index)
    return df.drop(columns=['kjønn']).join(encoded_df)

df = kjønn_dummies(df)

etnisitet_encoded = etnisitet_encoder.transform(df[['etnisitet']])
etnisitet_encoded_df = pd.DataFrame(etnisitet_encoded, columns=etnisitet_encoder.get_feature_names_out(['etnisitet']), index=df.index)
df = df.drop(columns=['etnisitet']).join(etnisitet_encoded_df)

sykdom_encoder = OneHotEncoder(sparse_output=False)
sykdom_encoded = sykdom_encoder.fit_transform(df[['sykdomskategori']])
sykdom_encoded_df = pd.DataFrame(sykdom_encoded, columns=sykdom_encoder.get_feature_names_out(), index=df.index)
df = df.drop(['sykdomskategori'], axis=1).join(sykdom_encoded_df)


In [8]:
df = df.loc[:, ~df.columns.duplicated()]

expected_features = model.feature_names_in_
df_final = df.reindex(columns=expected_features).fillna(0).infer_objects(copy=False)

predictions = model.predict(df_final)

df_oppholdstid = pd.DataFrame(pasient_ids)
df_oppholdstid['oppholdstid'] = predictions.round(0)

df_oppholdstid.to_csv("predictions.csv", index=False)
print("Predictions saved to 'predictions.csv'")

df_oppholdstid.head()


Predictions saved to 'predictions.csv'


/var/folders/ls/z_zr2hl926d1wx3bcxqkz7dc0000gn/T/ipykernel_17498/1948600760.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df.reindex(columns=expected_features).fillna(0).infer_objects(copy=False)


,pasient_id,oppholdstid
0,1,15.0
1,22,15.0
2,25,15.0
3,26,9.0
4,29,23.0
